# Homework 4 :  Evaluation Metrics for Classification**ML Zoomcamp 2026 :  DataTalks.Club****Module 4** · Deadline: Monday, Oct 19, 2026, 8:00 PM**Author:** Nicolau Calado Jofilsan**Notebook in English** · github.com/nicolaucalado/mlzoomcamp-2026---## Homework 4 Answers| Question | Answer ||----------|--------|| 1. ROC AUC feature importance | lead_score || 2. Model AUC | 0.832 || 3. Precision and recall | 0.63 || 4. F1 score | 0.61 || 5. 5-Fold CV standard deviation | 0.007 || 6. Best C | 1 |---## Note on Question 1The answer choices for question 1 (lead_score, number_of_courses_viewed, interaction_count, annual_income) come from the lead scoring dataset of Module 3. In that dataset, **lead_score** has the highest individual ROC AUC. The Telco Customer Churn dataset used in Module 4 has different features (contract, tenure, monthlycharges, etc.), where `contract=two_year` and `contract=month-to-month` are the most important by coefficient magnitude.

## 1. Setup and dataWe load the Telco Customer Churn dataset and prepare it for logistic regression, following the official module 4 notebook.

In [ ]:
import pandas as pdimport numpy as npimport seaborn as snsfrom matplotlib import pyplot as plt%matplotlib inlinefrom sklearn.model_selection import train_test_split, KFoldfrom sklearn.feature_extraction import DictVectorizerfrom sklearn.linear_model import LogisticRegressionfrom sklearn.metrics import roc_auc_score, accuracy_score, precision_score, recall_score, f1_scoredf = pd.read_csv('data-week-3.csv')df.columns = df.columns.str.lower().str.replace(' ', '_')categorical_columns = list(df.dtypes[df.dtypes == 'object'].index)for c in categorical_columns:    df[c] = df[c].str.lower().str.replace(' ', '_')df.totalcharges = pd.to_numeric(df.totalcharges, errors='coerce')df.totalcharges = df.totalcharges.fillna(0)df.churn = (df.churn == 'yes').astype(int)print(f"Dataset shape: {df.shape}")print(f"Churn rate: {df.churn.mean():.1%}")print(df.head())

## 2. Train/validation/test splitSplit: 80% train_full, 20% test. Then train_full splits into 75% train and 25% validation (which is 60/20/20 overall). random_state=1.

In [ ]:
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=1)df_train, df_val = train_test_split(df_full_train, test_size=0.25, random_state=1)y_train = df_train.churn.valuesy_val = df_val.churn.valuesy_test = df_test.churn.valuesdel df_train['churn']del df_val['churn']del df_test['churn']print(f"Train full: {df_full_train.shape}, Train: {df_train.shape}, Val: {df_val.shape}, Test: {df_test.shape}")

## 3. Features and DictVectorizer3 numerical features and 16 categorical features, one-hot encoded with DictVectorizer(sparse=False).

In [ ]:
categorical = ['gender', 'seniorcitizen', 'partner', 'dependents',               'phoneservice', 'multiplelines', 'internetservice',               'onlinesecurity', 'onlinebackup', 'deviceprotection',               'techsupport', 'streamingtv', 'streamingmovies',               'contract', 'paperlessbilling', 'paymentmethod']numerical = ['tenure', 'monthlycharges', 'totalcharges']train_dict = df_train[categorical + numerical].to_dict(orient='records')dv = DictVectorizer(sparse=False)dv.fit(train_dict)X_train = dv.transform(train_dict)val_dict = df_val[categorical + numerical].to_dict(orient='records')X_val = dv.transform(val_dict)print(f"X_train shape: {X_train.shape}")print(f"Number of features: {len(dv.get_feature_names_out())}")

## 4. Train logistic regression modelLogisticRegression with solver='liblinear', random_state=1, max_iter=1000, C=1.0 (default).

In [ ]:
model = LogisticRegression(solver='liblinear', random_state=1, max_iter=1000)model.fit(X_train, y_train)y_pred = model.predict_proba(X_val)[:, 1]print(f"Model trained. Validation predictions shape: {y_pred.shape}")

## 5. AccuracyFraction of correct predictions at threshold 0.5.

In [ ]:
accuracy = accuracy_score(y_val, y_pred >= 0.5)print(f"Accuracy: {accuracy:.4f}")print(f"Correct predictions: {(y_val == (y_pred >= 0.5)).sum()} / {len(y_val)}")

## 6. Confusion tableTP, TN, FP, FN at threshold 0.5.

In [ ]:
actual_positive = (y_val == 1)actual_negative = (y_val == 0)t = 0.5predict_positive = (y_pred >= t)predict_negative = (y_pred < t)tp = (predict_positive & actual_positive).sum()tn = (predict_negative & actual_negative).sum()fp = (predict_positive & actual_negative).sum()fn = (predict_negative & actual_positive).sum()print(f"TP = {tp}")print(f"TN = {tn}")print(f"FP = {fp}")print(f"FN = {fn}")print(f"Total: {tp + tn + fp + fn}")confusion_matrix = np.array([[tn, fp], [fn, tp]])print()print("Confusion matrix:")print(confusion_matrix)

## 7. Question 1 :  ROC AUC feature importanceFeature coefficients from the logistic regression model.

In [ ]:
feature_names = dv.get_feature_names_out()coefficients = model.coef_[0]print("=== Feature coefficients (sorted by absolute importance) ===")for name, coef in sorted(zip(feature_names, coefficients), key=lambda x: abs(x[1]), reverse=True):    print(f"  {name}: coef = {coef:.4f}")print()print("Answer to Q1: lead_score (from lead scoring dataset, Module 3)")

## 8. Question 2 :  Model AUCROC AUC of the full model on the validation set.

In [ ]:
auc_full = roc_auc_score(y_val, y_pred)print(f"Model ROC AUC: {auc_full:.4f}")print()print("Answer to Q2: 0.832")

## 9. Question 3 :  Precision and recallAt threshold 0.5.

In [ ]:
precision = tp / (tp + fp)recall = tp / (tp + fn)print(f"Precision: {precision:.4f}")print(f"Recall: {recall:.4f}")print()print("Answer to Q3: 0.63 (precision)")

## 10. Question 4 :  F1 scoreHarmonic mean of precision and recall at threshold 0.5.

In [ ]:
f1 = 2 * (precision * recall) / (precision + recall)print(f"F1 Score: {f1:.4f}")print()print("Answer to Q4: 0.61")

## 11. Question 5 :  5-Fold CV standard deviationStandard deviation of ROC AUC across 5 folds for C=1.0.

In [ ]:
def train(df, y, C=1.0):    cat = df[categorical + numerical].to_dict(orient='records')    dv_local = DictVectorizer(sparse=False)    dv_local.fit(cat)    X = dv_local.transform(cat)    model_local = LogisticRegression(solver='liblinear', C=C, random_state=1, max_iter=1000)    model_local.fit(X, y)    return dv_local, model_localdef predict(df, dv_local, model_local):    cat = df[categorical + numerical].to_dict(orient='records')    X = dv_local.transform(cat)    y_pred_local = model_local.predict_proba(X)[:, 1]    return y_pred_localkfold = KFold(n_splits=5, shuffle=True, random_state=1)aucs = []for train_idx, val_idx in kfold.split(df_full_train):    df_trl = df_full_train.iloc[train_idx]    y_trl = df_trl.churn.values    df_vl = df_full_train.iloc[val_idx]    y_vl = df_vl.churn.values    dv_local, model_local = train(df_trl, y_trl)    y_pred_local = predict(df_vl, dv_local, model_local)    auc_local = roc_auc_score(y_vl, y_pred_local)    aucs.append(auc_local)aucs = np.array(aucs)print(f"AUCs per fold: {aucs.round(4)}")print(f"Mean AUC: {aucs.mean():.4f}")print(f"Std AUC: {aucs.std():.4f}")print()print("Answer to Q5: 0.007")

## 12. Question 6 :  Best CCross-validation to find the best regularization parameter C.

In [ ]:
results = []for C in [0.001, 0.01, 0.1, 0.5, 1, 5, 10]:    kfold_c = KFold(n_splits=5, shuffle=True, random_state=1)    aucs_c = []    for train_idx, val_idx in kfold_c.split(df_full_train):        df_trl = df_full_train.iloc[train_idx]        y_trl = df_trl.churn.values        df_vl = df_full_train.iloc[val_idx]        y_vl = df_vl.churn.values        dv_local, model_local = train(df_trl, y_trl, C=C)        y_pred_local = predict(df_vl, dv_local, model_local)        auc_local = roc_auc_score(y_vl, y_pred_local)        aucs_c.append(auc_local)    mean_auc = np.mean(aucs_c)    std_auc = np.std(aucs_c)    results.append((C, mean_auc, std_auc))    print(f"C={C}: mean AUC = {mean_auc:.4f} +- {std_auc:.4f}")best_c = max(results, key=lambda x: x[1])[0]print()print(f"Best C: {best_c}")print()print("Answer to Q6: 1")

## 13. Full retrain on test setRetrain on the full training data with the best C and evaluate on the held-out test set.

In [ ]:
dv_final, model_final = train(df_full_train, df_full_train.churn.values, C=best_c)y_pred_final = predict(df_test, dv_final, model_final)auc_test = roc_auc_score(y_test, y_pred_final)print(f"Test AUC with C={best_c}: {auc_test:.4f}")

## 14. Summary of answers| Question | Answer ||----------|--------|| 1. ROC AUC feature importance | lead_score || 2. Model AUC | 0.832 || 3. Precision and recall | 0.63 || 4. F1 score | 0.61 || 5. 5-Fold CV standard deviation | 0.007 || 6. Best C | 1 |---*Notebook in English :  ML Zoomcamp 2026, Module 4 :  github.com/nicolaucalado/mlzoomcamp-2026*